# Notebook 06 — Shoreline Positions Along Transects

This notebook records where the 2024 shoreline intersects each transect.
These positions form the baseline for future multi-year change analysis.
No change rates are calculated here.
The coastal NoData QA issue remains unresolved.

In [6]:
import sys
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

cwd = Path.cwd()
if (cwd / "notebooks").exists():
    project_root = cwd
elif (cwd.parent / "notebooks").exists():
    project_root = cwd.parent
else:
    project_root = cwd

PROJECT_ROOT = project_root
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SHORELINE_PATH = (
    PROCESSED_DIR / "mombasa_test" / "2024" / "shoreline_2024_shared_boundary_test.gpkg"
)
TRANSECTS_PATH = (
    PROJECT_ROOT / "outputs" / "transects" / "2024" / "shoreline_transects_2024.gpkg"
)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "positions" / "2024"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Shoreline:", SHORELINE_PATH)
print("Transects:", TRANSECTS_PATH)
print("Output:", OUTPUT_DIR)

Shoreline: [redacted env]\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\2024\shoreline_2024_shared_bo
undary_test.gpkg
Transects: [redacted env]\Documents\Data Science Projects\Shoremonitor\outputs\transects\2024\shoreline_transects_2024.gpkg
Output: [redacted env]\Documents\Data Science Projects\Shoremonitor\outputs\positions\2024


In [7]:
if not SHORELINE_PATH.exists():
    raise FileNotFoundError(f"Missing input: {SHORELINE_PATH}")
if not TRANSECTS_PATH.exists():
    raise FileNotFoundError(f"Missing input: {TRANSECTS_PATH}")

shoreline = gpd.read_file(SHORELINE_PATH)
transects = gpd.read_file(TRANSECTS_PATH)

print("Shoreline CRS:", shoreline.crs, "count:", len(shoreline))
print("Transects CRS:", transects.crs, "count:", len(transects))
print("Shoreline bounds:", shoreline.total_bounds)
print("Transects bounds:", transects.total_bounds)

if shoreline.empty:
    raise ValueError("Shoreline candidates are empty")
if transects.empty:
    raise ValueError("Transects are empty")
if shoreline.crs != transects.crs:
    raise ValueError(f"CRS mismatch: {shoreline.crs} vs {transects.crs}")

Shoreline CRS: EPSG:32737 count: 15802
Transects CRS: EPSG:32737 count: 23
Shoreline bounds: [ 572150. 9546790.  594340. 9568830.]
Transects bounds: [ 571264.72561532 9552120.          576220.         9556180.        ]


In [8]:
shoreline_union = shoreline.geometry.unary_union
records = []
intersection_id = 0

for _, trow in transects.iterrows():
    tid = trow.get("transect_id", trow.name)
    inter = trow.geometry.intersection(shoreline_union)
    pts = []
    if inter.is_empty:
        pts = []
    elif inter.geom_type == "Point":
        pts = [inter]
    elif inter.geom_type == "MultiPoint":
        pts = list(inter.geoms)
    elif hasattr(inter, "geoms"):
        for g in inter.geoms:
            if g.geom_type == "Point":
                pts.append(g)
    if not pts:
        # record as missing
        records.append(
            {
                "transect_id": tid,
                "intersection_id": None,
                "geometry": None,
                "status": "missing",
            }
        )
        continue
    for p in pts:
        intersection_id += 1
        records.append(
            {
                "transect_id": tid,
                "intersection_id": intersection_id,
                "geometry": p,
                "status": "found",
            }
        )

positions = gpd.GeoDataFrame(records, crs=shoreline.crs)
found = positions[positions["status"] == "found"].copy()
out_gpkg = OUTPUT_DIR / "shoreline_positions_2024.gpkg"
found.to_file(out_gpkg, driver="GPKG", layer="shoreline_positions")
print("Intersections found:", len(found))
print("Saved:", out_gpkg)

[redacted env]\Temp\ipykernel_20684\1452921982.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'uni
on_all()' method instead.
  shoreline_union = shoreline.geometry.unary_union


Intersections found: 114
Saved: [redacted env]\Documents\Data Science Projects\Shoremonitor\outputs\positions\2024\shoreline_positions_2024.gpkg


In [9]:
found["longitude"] = found.geometry.x
found["latitude"] = found.geometry.y
summary = found[["transect_id", "intersection_id", "longitude", "latitude"]].copy()
out_csv = OUTPUT_DIR / "shoreline_positions_2024.csv"
summary.to_csv(out_csv, index=False)
print("Saved:", out_csv)
print(summary.head(10).to_string(index=False))

# transect intersection counts
counts = positions.groupby("transect_id").apply(
    lambda d: (d["status"] == "found").sum()
)
n0 = int((counts == 0).sum())
n1 = int((counts == 1).sum())
nm = int((counts > 1).sum())
print(f"Transects with 0 intersections: {n0}")
print(f"Transects with 1 intersection: {n1}")
print(f"Transects with multiple intersections: {nm}")

Saved: [redacted env]\Documents\Data Science Projects\Shoremonitor\outputs\positions\2024\shoreline_positions_2024.csv
transect_id  intersection_id     longitude     latitude
      T0000                1 574670.000000 9.555260e+06
      T0000                2 574700.000000 9.555170e+06
      T0000                3 574703.333333 9.555160e+06
      T0000                4 574956.666667 9.554400e+06
      T0001                5 574485.746635 9.555280e+06
      T0001                6 574485.746635 9.555180e+06
      T0001                7 574485.746635 9.554880e+06
      T0002                8 574090.000000 9.554248e+06
      T0002                9 574036.238267 9.554170e+06
      T0002               10 574510.000000 9.554861e+06
Transects with 0 intersections: 0
Transects with 1 intersection: 2
Transects with multiple intersections: 21


In [10]:
import folium

center = [transects.total_bounds[1], transects.total_bounds[0]]
m = folium.Map(location=center, zoom_start=12)
for _, row in transects.iterrows():
    coords = (
        [(y, x) for x, y in row.geometry.coords]
        if hasattr(row.geometry, "coords")
        else []
    )
    folium.PolyLine(coords, color="blue", weight=2).add_to(m)
for _, row in found.iterrows():
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x], radius=4, color="red", fill=True
    ).add_to(m)

map_path = OUTPUT_DIR / "shoreline_positions_map.html"
m.save(str(map_path))
print("Map saved:", map_path)

Map saved: [redacted env]\Documents\Data Science Projects\Shoremonitor\outputs\positions\2024\shoreline_positions_map.html


2024 positions are the baseline dataset.
Change rates require additional years of shoreline positions.
The unresolved coastal NoData QA issue remains.